In [1]:
#!pip install line-bot-sdk

In [2]:
LINE_CHANNEL_ACCESS_TOKEN ='2/dhY9KZ26CCFtkQn7GYlbkHXExlfF4VTandkH7w1C4cTicq67dKbfkmc7BD6RUv5oaUw3bLDV2pkTGjQf35wXTNSpHdDG6n3WL5qhkr8ZY3kJJNqqYiwQf3yV5qn8exV398q3I6vL1K9gKbvBQRxQdB04t89/1O/w1cDnyilFU='
LINE_CHANNEL_SECRET = 'e9a682fa8143239086413e870e5865b3'#'Basic settings的Channel Secret'
USER_ID='Udb111becb66f5038ed5e8fef3a558ece'
YOUR_MESSAGE=' XDDD'

In [3]:
from linebot import LineBotApi
from linebot.exceptions import LineBotApiError
from linebot.models import TextSendMessage, FlexSendMessage

line_bot_api = LineBotApi(LINE_CHANNEL_ACCESS_TOKEN)
user_id = USER_ID


def sendLineMessage(msg):
    """
    msg 可以是：
    - str：純文字
    - dict：當作 Flex contents 直接送
    """
    try:
        if isinstance(msg, dict):
            message = FlexSendMessage(
                alt_text='表格訊息',
                contents=msg
            )
        else:
            message = TextSendMessage(text=str(msg))

        line_bot_api.push_message(user_id, message)
        print('消息发送成功！')

    except LineBotApiError as e:
        print('消息发送失败:', e)


C:\Users\user\AppData\Local\Temp\ipykernel_37016\2513558176.py:5: LineBotSdkDeprecatedIn30: Call to deprecated class LineBotApi. (Use v3 class; linebot.v3.<feature>. See https://github.com/line/line-bot-sdk-python/blob/master/README.rst for more details.) -- Deprecated since version 3.0.0.
  line_bot_api = LineBotApi(LINE_CHANNEL_ACCESS_TOKEN)


In [9]:
import json

def build_flex_table_from_json(
    raw,
    title="表格資料",
    max_rows=10,
    url_fields=("url", "link", "href")
):
    """
    raw 可以是：
    - list[dict]
    - JSON 字串（會用 json.loads() 解析）

    功能：
    - 自動用第一筆 dict 的 key 當欄位順序
    - 自動偵測 url/link/href 欄位，整列變成可點擊
    - 預設隱藏 URL 欄位不顯示在表格中（只用來當 action）
    """

    # 若是字串，就先轉成 Python 物件
    if isinstance(raw, str):
        data = json.loads(raw)
    else:
        data = raw

    # 確保是 list
    if not isinstance(data, list) or not data:
        raise ValueError("資料格式錯誤，需要非空的 list[dict]")

    # 取第一筆的 key 當欄位
    first = data[0]
    if not isinstance(first, dict):
        raise ValueError("資料格式錯誤，list 內元素必須是 dict")

    all_columns = list(first.keys())

    # 要不要把 url 欄位藏起來（通常藏掉就好）
    visible_columns = [c for c in all_columns if c not in url_fields]

    # 如果資料太多，先截斷
    rows = data[:max_rows]

    # ===== 表頭列 =====
    header_box = {
        "type": "box",
        "layout": "horizontal",
        "spacing": "sm",
        "contents": []
    }
    for col in visible_columns:
        header_box["contents"].append({
            "type": "text",
            "text": str(col),
            "weight": "bold",
            "size": "xs",
            "flex": 1,
            "wrap": False
        })

    # ===== 資料列 =====
    row_boxes = []
    for row in rows:
        # 找這一列有沒有 URL
        row_url = None
        for uf in url_fields:
            if uf in row and row[uf]:
                row_url = str(row[uf])
                break

        row_box = {
            "type": "box",
            "layout": "horizontal",
            "spacing": "sm",
            "contents": []
        }

        # 如果有 URL，整列可點擊
        if row_url:
            row_box["action"] = {
                "type": "uri",
                "uri": row_url
            }

        # 填入欄位內容
        for col in visible_columns:
            value = row.get(col, "")
            text = str(value)

            # 簡單數字偵測：如果都是數字/小數/百分比，就右對齊
            is_number_like = False
            stripped = text.replace(",", "").replace("%", "")
            #if stripped.replace(".", "", 1).isdigit():
            #    is_number_like = True

            row_box["contents"].append({
                "type": "text",
                "text": text,
                "size": "xs",
                "flex": 1,
                "wrap": True,
                "align": "end" if is_number_like else "start"
            })

        row_boxes.append(row_box)

    # ===== 組成 Flex Bubble =====
    contents = {
        "type": "bubble",
        "body": {
            "type": "box",
            "layout": "vertical",
            "spacing": "md",
            "contents": [
                {
                    "type": "text",
                    "text": title,
                    "weight": "bold",
                    "size": "md"
                },
                {"type": "separator", "margin": "md"},
                header_box,
                {"type": "separator", "margin": "sm"},
                *row_boxes
            ]
        }
    }

    return contents


# Test

sendLineMessage('aaa')

In [10]:
data = [
    {"項目": "高度", "數值": "178", "單位": "cm", "url": "https://example.com/height"},
    {"項目": "體重", "數值": "75", "單位": "kg", "url": "https://example.com/weight"},
    {"項目": "體脂", "數值": "18.5", "單位": "%", "url": "https://example.com/bodyfat"}
]
flex_contents = build_flex_table_from_json(data, title="測試表格（可點擊）")
sendLineMessage(flex_contents)

消息发送成功！


C:\Users\user\AppData\Local\Temp\ipykernel_37016\2513558176.py:24: LineBotSdkDeprecatedIn30: Call to deprecated method push_message. (Use 'from linebot.v3.messaging import MessagingApi' and 'MessagingApi(...).push_message(...)' instead. See https://github.com/line/line-bot-sdk-python/blob/master/README.rst for more details.) -- Deprecated since version 3.0.0.
  line_bot_api.push_message(user_id, message)
